In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
import time
import sib_api_v3_sdk
import re
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime,timezone
import uuid 


In [3]:
load_dotenv(override=True)

True

In [4]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
# json extractor
def extract_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print("Still failed:", e)

    return None

In [6]:
def safe_generate(prompt):
    max_retries = 3

    for _ in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "Return only valid JSON"},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7
            )

            return response.choices[0].message.content

        except Exception as e:
            if "429" in str(e):
                print("⏳ Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
            else:
                print("Error:", e)
                return None

    print("Max retries reached")
    return None

In [18]:
def run_agent(lead):
    full_prompt = f"""
You are an elite SDR at Hamro Aadhiyan writing hyper-personalized cold emails.

Your job is NOT to write marketing copy.
Your job is to write emails that feel like a real human noticed this specific student and wrote a short message.

---

COMPANY CONTEXT:
- BSC.CSIT notes, videos, past questions
- AI-based weak area detection for exam improvement

LEAD:
- Name: {lead['name']}
- Role: {lead['role']}
- Goal: {lead['goal']}

SENDER:
- Name: Prasanna Niroula
- Position: CEO, Hamro Aadhiyan
- Location: Biratnagar-4, Kanchanbari
- Website: www.hamroaadhiyan.com

---

⚠️ STRICT PERSONALIZATION RULES:
- NEVER use generic phrases like:
  "we understand the importance"
  "we are committed to helping"
  "our comprehensive solutions"
- DO NOT start with cliché corporate intros
- FIRST line must feel human and specific to the student
- MUST naturally reference their goal: {lead['goal']}
- MUST sound like a real student-to-student helpful message
- Keep tone natural, not salesy

---

EMAIL TYPES:
1. Professional → human, simple, not corporate
2. Humorous → light, relatable, conversational (no forced jokes)
3. Concise → max 3–5 short lines, direct impact

---

HTML RULES:
- Use ONLY inline CSS
- Use <div> layout (no full <html> required)
- max-width: 600px centered card
- background: #f4f6f8
- card: white with padding 20px, border-radius 10px

TEXT STYLING:
- Greeting: font-size 20px, font-weight bold
- Body: font-size 15px, line-height 1.6
- Signature: font-size 13px, color #666

---

OUTPUT RULES:
- NO markdown
- NO explanation
- NO extra text
- ONLY valid JSON

FORMAT:
{{
  "emails": {{
    "professional": "<div>...</div>",
    "humorous": "<div>...</div>",
    "concise": "<div>...</div>"
  }}
}}
"""

    response = safe_generate(full_prompt)

    if response is None:
        print("Generation failed completely")
        return None

    print(response)

    data = extract_json(response)

    if data is None:
        print("JSON parsing failed")
        print("RAW OUTPUT:\n", response)
        return None

    return data

In [19]:
def evaluate_emails(emails):
    prompt = f"""
You are an expert SDR email formatter.

You must select the BEST email and return it EXACTLY as-is.

CRITICAL RULES:
- Output must be CLEAN HTML email only
- NO placeholders like "(button here)" or "..."  
- NO explanations or comments
- NO markdown
- NO extra text outside email
- Preserve structure exactly as given
- If email is not HTML, convert it into proper HTML

Emails:

Professional:
{emails['professional']}

Humorous:
{emails['humorous']}

Concise:
{emails['concise']}

Return ONLY JSON:

{{
  "final_email": "<clean full HTML email>"
}}
"""

    response = safe_generate(prompt)

    if response is None:
        print("Evaluation failed")
        return None

    print("RAW RESPONSE:", response)

    data = extract_json(response)

    if data is None:
        print("JSON parsing failed")
        return None

    return data

In [20]:
# send_email function for dynamically sending email
def send_email(to_email,subject, body):
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key['api-key']= os.getenv("BREVO_API_KEY")

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    ) 
    email = sib_api_v3_sdk.SendSmtpEmail(
        to=[{"email":to_email}],
        sender={
            "name":"Hamro Aadhiyan",
            "email":"anything@prasannaniroula.com.np"
        },
        subject= subject,
        html_content = body
    )
    try:
        response = api_instance.send_transac_email(email)
        print("Email sent via Brevo:",response)
        return response
    except Exception as e:
        print("Error sending the emails:",e)
        return None

In [21]:
# storing record of email forwarded
def store_record(data):
    with open("email_logs.ndjson","a") as f:
        f.write(json.dumps(data)+"\n")

In [22]:
def now():
    return datetime.now(timezone.utc).isoformat()


In [23]:
def create_email_record(lead, to_email, subject, body):
    return{
        "id":str(uuid.uuid4()),
        "name":lead["name"],
        "email":to_email,
        "subject":subject,
        "body":body,
        "status": {
            "sent": False,
            "delivered": False,
            "opened": False,
            "replied": False,
            "message_id": None
        },
        "timestamps": {
            "sent": None,
            "delivered": None,
            "opened": None,
            "replied": None
}
    }


In [24]:
# Tools
def run_sdr(lead, to_email):
    #generating emails
    result= run_agent(lead)
    if not result:
        print("Email generation failed")
        return
    
    emails = result["emails"]
    print(emails)

    #Emails selection
    result = evaluate_emails(emails)
    if not result:
        print("Evaluation failed")
        return
    
    final_email = result["final_email"]
    subject = f"Quick help for {lead['role']}"

    # Recording email 
    record = create_email_record(lead, to_email, subject, final_email)

    #sending email
    response = send_email(to_email = to_email , subject=subject, body= final_email)
    print(final_email)

    if response:
        record["status"]["sent"] = True
        record["timestamps"]["sent"]= now()
        record["status"]["message_id"] = response.message_id
        print("Email sent successfully")
    else:
        record["status"]["sent"]=False
        print("Email sending failed")
    
    #storing record
    store_record(record)






In [27]:
lead = {
    "name": "Prasanna",
    "role": "BSC.CSIT student",
    "goal": "improve exam performance",
}

run_sdr(
    lead=lead,
    to_email="prasannaniroula987@gmail.com"
)

{
  "emails": {
    "professional": "<div style='max-width: 600px; margin: 0 auto; background: #f4f6f8; padding: 0;'><div style='background: white; padding: 20px; border-radius: 10px;'><span style='font-size: 20px; font-weight: bold;'>Hi Prasanna,</span><br><span style='font-size: 15px; line-height: 1.6;'>I noticed you're a BSC.CSIT student looking to improve your exam performance. I've been in your shoes before and I know how tough it can get. That's why I wanted to reach out and see if our AI-based weak area detection could help you identify areas where you need to focus.</span><br><span style='font-size: 13px; color: #666;'>Best, Prasanna Niroula</span></div></div>",
    "humorous": "<div style='max-width: 600px; margin: 0 auto; background: #f4f6f8; padding: 0;'><div style='background: white; padding: 20px; border-radius: 10px;'><span style='font-size: 20px; font-weight: bold;'>Hey Prasanna,</span><br><span style='font-size: 15px; line-height: 1.6;'>Let's face it, exams can be a rea